# 数据分析与机器学习

各节为独立示例，可按需阅读和运行。

目录：

- NumPy 入门与 ndarray
- NumPy 数组创建
- NumPy 数组变换
- NumPy 索引、切片与广播
- NumPy 聚合操作
- NumPy 合并、拆分、插入与删除
- NumPy 练习
- NumPy meshgrid 与 indices
- NumPy where、argmin 与 unravel_index
- NumPy 排序
- NumPy 函数工具
- Pandas 入门
- Pandas Series
- Pandas DataFrame 基础
- Pandas 多级索引
- Pandas 数据合并
- Pandas 数据读取与清洗
- Pandas 可视化
- Matplotlib 折线图
- Matplotlib 坐标轴、刻度与标注
- Matplotlib 子图
- Matplotlib 散点图
- Matplotlib 柱形图
- Matplotlib 直方图
- Matplotlib 饼图
- Matplotlib 误差可视化
- Matplotlib 轮廓图
- Matplotlib 三维图形
- scikit-learn 现代建模流程

## NumPy 入门与 ndarray

NumPy 是 Python 科学计算的基础库，核心对象是多维数组 `ndarray`。它把数据和形状、维度、数据类型等元数据放在一起，并通过向量化计算减少显式 Python 循环。

### 安装

```bash
pip install numpy
```

### ndarray 属性

常用属性包括 `ndim`、`shape`、`size`、`dtype` 和 `itemsize`。理解 `axis` 很关键：一维数组只有 `axis=0`，二维数组通常把行方向看作 `axis=0`，列方向看作 `axis=1`。

In [ ]:
import numpy as np


a = np.array([1, 2, 3])
b = np.array([[1, 2, 3], [4, 5, 6]])
c = np.array([[[1, 2, 3], [4, 5, 6]]])

for name, array in {"a": a, "b": b, "c": c}.items():
    print(name, "ndim=", array.ndim, "shape=", array.shape, "dtype=", array.dtype)

### 向量化

NumPy 的数组运算会把整块数据交给底层实现处理。下面的例子展示同一个倒数计算，显式循环和数组表达式的写法差异。

In [ ]:
rng = np.random.default_rng(42)
values = rng.integers(1, 100, size=10)


def compute_reciprocals(numbers: np.ndarray) -> np.ndarray:
    output = np.empty(len(numbers), dtype=float)
    for index in range(len(numbers)):
        output[index] = 1.0 / numbers[index]
    return output


print(compute_reciprocals(values))
print(1.0 / values)

## NumPy 数组创建

数组可以从 Python 序列创建，也可以用 NumPy 提供的构造函数批量生成。这个主题重点关注 `array`、`arange`、`linspace`、全零全一数组、重复数组和随机数。

### 从序列创建

`np.array()` 可以从列表、元组或嵌套序列创建数组，并可以通过 `dtype` 指定元素类型。

In [ ]:
import numpy as np


print(np.array([1, 2, 3], dtype=np.float32))
print(np.array((1, 2, 3)))
print(np.array([[1, 2], [3, 4], (5, 6)]))

### 批量生成

`arange()` 类似内置 `range()`，`linspace()` 按指定数量生成等间隔数据。`zeros()`、`ones()`、`full()` 和 `eye()` 适合初始化固定形状的数组。

In [ ]:
print(np.arange(1, 11, dtype=float))
print(np.linspace(1, 10, 6, endpoint=True))
print(np.zeros((2, 3)))
print(np.ones((2, 3)))
print(np.full((2, 3), 7))
print(np.eye(3))

### 重复与随机数

`tile()` 重复整体结构，`repeat()` 重复单个元素。随机数推荐使用 `np.random.default_rng()` 创建生成器。

In [ ]:
a = np.array([1, 2, 3])
print("tile:", np.tile(a, (2, 2)))
print("repeat:", np.repeat(a, 2))

rng = np.random.default_rng(42)
print(rng.integers(1, 10, size=(2, 3)))
print(rng.normal(0, 1, size=5))

## NumPy 数组变换

数组变换包括形状变换、轴交换、展平和类型转换。判断返回值是视图还是副本，是写 NumPy 代码时需要特别留意的点。

### 形状与轴

`reshape()` 返回新视图或副本，`resize()` 修改原数组，`flatten()` 返回一维副本，`ravel()` 通常返回一维视图。

In [ ]:
import numpy as np


a = np.arange(24)
print(a.reshape(2, 3, 4))

a.resize(2, 3, 4)
print(a.shape)
print(a.swapaxes(0, 2).shape)
print(a.transpose(2, 1, 0).shape)

In [ ]:
a = np.array([(1, 2, 3, 4), (3, 1, 4, 2)])
flat = a.flatten()
raveled = a.ravel()

print(flat)
print(raveled)

### 数据类型

数组通常要求元素类型一致。可以在创建时指定 `dtype`，也可以用 `astype()` 转换类型。若确实需要混合对象，可以使用 `dtype=object`。

In [ ]:
list2d = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
arr2d_f = np.array(list2d, dtype=np.float32)
arr2d_i = arr2d_f.astype("int")
arr2d_s = arr2d_i.astype("str")
arr1d_obj = np.array([1, "a"], dtype="object")

print(arr2d_f.dtype)
print(arr2d_i)
print(arr2d_s)
print(type(arr1d_obj.tolist()))

## NumPy 索引、切片与广播

NumPy 的数据访问包括普通切片、花式索引和布尔索引。广播机制则决定不同形状数组能否一起参与计算。

### 索引和切片

普通切片通常返回视图，修改切片可能影响原数组；花式索引通常返回副本。

In [ ]:
import numpy as np


a = np.arange(1, 6)
view = a[2:4]
view[0] = 10
print("slice view:", a, view)

a = np.arange(1, 6)
copy = a[[2, 3]]
copy[0] = 10
print("fancy copy:", a, copy)

### 布尔索引

布尔表达式会生成同形状的布尔数组，然后用于筛选满足条件的元素或行列。

In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print(a[a > 5])
print(a[a[:, 0] >= 4])

### 广播规则

广播从形状尾部向前比较：两个维度相等，或其中一个维度为 1 时兼容。低维数组可以在前面补长度为 1 的轴。

In [ ]:
a = np.arange(1, 16).reshape(3, 5)
b = np.array([2, 3, 4, 5, 6])
print(a + b)

A = np.zeros((2, 3, 4))
B = np.zeros((3, 1))
print((A + B).shape)

### 元素级函数

`abs`、`sqrt`、`exp`、`log`、三角函数等通用函数会逐元素作用于数组。下面用 `exp` 实现一个简单 softmax。

In [ ]:
x = np.array([3, 1, -3], dtype=np.float32)
exp_x = np.exp(x)
print(exp_x / exp_x.sum())

## NumPy 聚合操作

聚合操作把一组值归约为统计结果，例如最大值、最小值、求和、均值、方差、中位数和百分位数。多数聚合函数都支持 `axis` 参数。

### 一维聚合

常见函数包括 `sum`、`mean`、`std`、`var`、`min`、`max`、`median` 和 `percentile`。许多函数还有 `nan*` 版本用于跳过缺失值。

In [ ]:
import numpy as np


a = np.array([1, 2, 3, 4, 5])
print(a.max())
print(np.percentile(a, q=[0, 25, 50, 75, 100]))
print(np.mean(a), np.std(a))

### 按轴聚合

二维数组中，`axis=0` 按列聚合，`axis=1` 按行聚合。

In [ ]:
matrix = np.arange(1, 13).reshape(3, 4)
print(matrix)
print(matrix.max(axis=0))
print(matrix.max(axis=1))
print(matrix.mean(axis=0))

### argmin、argmax 与布尔聚合

高维数组上的 `argmin()` 和 `argmax()` 默认返回展平后的索引，可以用 `unravel_index()` 转回多维坐标。`any()` 和 `all()` 也支持按轴判断。

In [ ]:
rng = np.random.default_rng(42)
a = rng.integers(1, 10, (3, 4))
flat_index = np.argmin(a)
print(a)
print(flat_index)
print(np.unravel_index(flat_index, a.shape))
print((a > 5).any(axis=1))
print((a > 5).all(axis=0))

## NumPy 合并、拆分、插入与删除

数组拼接需要先明确目标轴和结果形状。`concatenate()` 沿既有轴拼接，`stack()` 会新增轴，`split()` 则执行相反操作。

### 合并

常用函数包括 `concatenate()`、`stack()`、`hstack()`、`vstack()`、`dstack()`、`row_stack()` 和 `column_stack()`。

In [ ]:
import numpy as np


a = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([[7, 8, 9], [11, 12, 13]])

print(np.concatenate([a, b], axis=0))
print(np.concatenate([a, b], axis=1))
print(np.stack([a, b], axis=0).shape)
print(np.vstack([a, b]))
print(np.hstack([a, b]))

### append 和 pad

`append()` 默认会先展平再拼接。需要给数组边界补值时，`pad()` 往往比反复 append 更清晰。

In [ ]:
print(np.append(a, b))
print(np.append(a, b, axis=0))

base = np.arange(1, 16).reshape(3, 5)
print(np.pad(base, 1, mode="edge"))

### 拆分、插入和删除

`split()` 可以按份数或索引拆分数组。`insert()` 与 `delete()` 支持指定轴；不指定轴时会先展平。

In [ ]:
x = np.arange(12)
print(np.split(x, 3))
print(np.split(x, [3, 8]))

matrix = np.arange(1, 10).reshape(3, 3)
print(np.insert(matrix, 1, [10, 10, 10], axis=0))
print(np.delete(matrix, 1, axis=1))

## NumPy 练习

这些练习来自原 NumPy 综合 notebook，集中训练数组构造、向量化、分段函数、按轴聚合、距离筛选和多元高斯密度。

### 练习 1：构造二维数组

创建一个 `m x n` 二维数组，使其元素满足 `a[i, j] = j - i`。

In [ ]:
import numpy as np


m, n = 2, 3
print(np.arange(n) - np.arange(m).reshape(-1, 1))

### 练习 2：向量化函数

给定一维数组 `x`，计算 `f(x) = x**2 + x + 1`。

In [ ]:
x = np.array([1, 2, 3, 4, 5])
print(x**2 + x + 1)

### 练习 3：分段函数

当 `x < 0` 时返回 `1 + x`，否则返回 `1 - x`。

In [ ]:
x = np.array([-1, -2, 0, 1, 2])
print(np.where(x >= 0, 1 - x, 1 + x))
print(np.piecewise(x, [x >= 0, x < 0], [lambda v: 1 - v, lambda v: 1 + v]))

### 练习 4：行列极差

给定二维数组，计算每一行和每一列最大值与最小值之差。

In [ ]:
a = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
print(a.max(axis=1) - a.min(axis=1))
print(a.max(axis=0) - a.min(axis=0))

### 练习 5：阈值内点的质心

给定点集 `a`、中心点 `p` 和阈值 `eps`，返回所有距离小于等于阈值的点的平均值。

In [ ]:
a = np.array([[2, 1], [1, 6], [1, 7], [7, 1], [9, 4]])
p = np.array([3, 3])
eps = 5
print(a[np.linalg.norm(a - p, axis=1) <= eps].mean(axis=0))

### 练习 6：多元高斯密度

输入 `X`、`mu` 和协方差矩阵 `sigma`，输出每个样本点对应的概率密度。

In [ ]:
def multi_gaussian(X: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    dim = sigma.shape[0]
    coef = 1.0 / np.sqrt((2 * np.pi) ** dim * np.linalg.det(sigma))
    inv_sigma = np.linalg.pinv(sigma)
    delta = X - mu
    exponent = -0.5 * np.sum(delta @ inv_sigma * delta, axis=1)
    return coef * np.exp(exponent)


X = np.array([[2, 1], [1, 3], [3, 3]])
mu = np.array([0, 0])
sigma = np.array([[1, 0], [0, 1]])
print(multi_gaussian(X, mu, sigma))

## NumPy meshgrid 与 indices

`np.meshgrid()` 根据一维坐标数组生成坐标网格，常用于等高线、三维曲面和分类边界可视化。

### meshgrid

二维情况下，`indexing="xy"` 返回笛卡尔坐标风格，`indexing="ij"` 返回矩阵索引风格。

In [ ]:
import numpy as np


x = np.linspace(1, 10, 5)
y = np.linspace(1, 10, 10)
xx, yy = np.meshgrid(x, y)
print(xx.shape, yy.shape)

i, j = np.meshgrid(np.arange(3), np.arange(2), indexing="ij")
print(i)
print(j)

### indices

如果只是生成矩阵索引，`np.indices()` 通常更直接，也适合提取固定大小的子数组。

In [ ]:
a = np.arange(12).reshape(3, 4)
i, j = np.indices((2, 2))
print(a)
print(a[i, j])
print(a[i + 1, j + 1])

### 可视化网格

网格坐标可以直接传给 Matplotlib 的等高线函数。

In [ ]:
import matplotlib.pyplot as plt


x = np.linspace(-10, 10, 25)
y = np.linspace(-10, 10, 25)
xx, yy = np.meshgrid(x, y)
zz = xx**2 + 4 * yy**2
plt.contourf(xx, yy, zz, cmap="rainbow")

## NumPy where、argmin 与 unravel_index

`np.where()` 可以返回满足条件的位置，也可以按条件在两个候选值之间选择。`argmin()` 和 `argmax()` 用于定位极值。

### 条件位置

只传入条件时，`np.where()` 返回索引元组；一维数组中常取第一个元素作为索引数组。

In [ ]:
import numpy as np


rng = np.random.default_rng(42)
a = rng.integers(0, 10, size=10)
positions = np.where(a > 5)
print(a)
print(positions)
print(positions[0])
print(a.take(positions))

### 条件选择

传入 `x` 和 `y` 时，条件为真返回 `x`，否则返回 `y`。

In [ ]:
print(np.where(a > 5, 1, 0))
print(np.where(a % 2 == 0, "even", "odd"))

### 极值位置

二维及以上数组的 `argmin()` 和 `argmax()` 默认返回展平后的索引，需要用 `np.unravel_index()` 转成多维坐标。

In [ ]:
matrix = rng.integers(1, 10, (3, 4))
flat_min = np.argmin(matrix)
flat_max = np.argmax(matrix)
print(matrix)
print(flat_min, np.unravel_index(flat_min, matrix.shape))
print(flat_max, np.unravel_index(flat_max, matrix.shape))

## NumPy 排序

NumPy 提供直接排序和间接排序。直接排序会重排值；间接排序返回索引，更适合保持行记录的整体关系。

### sort

`np.sort(a, axis=-1, kind=None, order=None)` 默认沿最后一个轴排序。`axis=None` 会先展平数组。

In [ ]:
import numpy as np


rng = np.random.default_rng(42)
arr = rng.integers(1, 6, size=(8, 4))
print(arr)
print(np.sort(arr, axis=0))
print(np.sort(arr, axis=1))

### argsort

`argsort()` 返回从小到大排序后的索引。对二维表格按某列排序时，用索引选择整行可以保留行内字段关系。

In [ ]:
x = np.array([1, 10, 5, 2, 8, 9])
index = np.argsort(x)
print(index)
print(x[index])

row_index = arr[:, 0].argsort()
print(arr[row_index])
print(arr[row_index[::-1]])

### lexsort

`np.lexsort()` 支持多键排序。传入的键从次要键到主要键排列，也就是最右侧键优先级最高。

In [ ]:
lex_index = np.lexsort((arr[:, 0], arr[:, 1], arr[:, 3]))
print(arr)
print(lex_index)
print(arr[lex_index])

## NumPy 函数工具

这个主题整理 `vectorize()`、`apply_along_axis()`、`piecewise()`、`digitize()` 和 `clip()`。这些函数适合把标量逻辑或分段规则应用到数组上。

### vectorize

`np.vectorize()` 可以把原本处理单个值的函数包装成逐元素处理数组的函数。它主要改善接口表达，不等同于底层向量化加速。

In [ ]:
import numpy as np


def foo(x: int) -> float:
    return x**2 if x % 2 == 1 else x / 2


foo_v = np.vectorize(foo, otypes=[float])
print(foo(10), foo(11))
print(foo_v([10, 11, 12]))
print(foo_v([[10, 11, 12], [1, 2, 3]]))

### apply_along_axis

`np.apply_along_axis(func1d, axis, arr)` 会沿指定轴取出一维切片，并把函数应用到每个切片。

In [ ]:
rng = np.random.default_rng(42)
a = rng.integers(1, 10, size=(4, 5))
print(a)
print(np.apply_along_axis(lambda x: x.max() - x.min(), axis=1, arr=a))
print(np.apply_along_axis(lambda x: x.max() - x.min(), axis=0, arr=a))

### piecewise、digitize 和 clip

`piecewise()` 根据条件列表应用不同函数，`digitize()` 返回值所属分箱位置，`clip()` 把值限制在上下界内。

In [ ]:
x = np.arange(0, 10)
print(np.piecewise(x, [x < 4, x >= 6], [lambda v: v**2, lambda v: v * 100]))

bins = np.array(range(-99, 102, 3))
print(np.digitize([-98, 68], bins))
print(np.clip(x, 3, 8))

## Pandas 入门

Pandas 是基于 NumPy 的表格数据分析库，适合处理混合类型数据、数据清洗、索引、合并和可视化。它的两个核心数据结构是 `Series` 和 `DataFrame`。

### 安装

```bash
pip install pandas
```

如果使用 Anaconda，也可以用 `conda install pandas`。

In [ ]:
import numpy as np
import pandas as pd


rng = np.random.default_rng(42)
print(pd.__name__)

### Series 和 DataFrame

`Series` 是带索引的一维数组，`DataFrame` 是按行和列组织的二维表格。Pandas 专门面向表格和混合类型数据，NumPy 更适合统一类型的数值数组。

In [ ]:
series = pd.Series([100, 200, 300], index=["USA", "France", "China"])
frame = pd.DataFrame(rng.integers(1, 10, size=(3, 2)), columns=["A", "B"])

print(series)
print(frame)

## Pandas Series

`Series` 是一维带标签数组，由数据和索引组成。创建 `Series` 时，`data` 可以来自列表、NumPy 数组或字典。

### 创建 Series

列表和 NumPy 数组会使用默认整数索引，或者使用显式传入的 `index`。

In [ ]:
import numpy as np
import pandas as pd


countries = ["USA", "France", "China"]
my_data = [100, 200, 300]
array = np.array(my_data)

print(pd.Series(data=my_data, index=countries))
print(pd.Series(data=my_data))
print(pd.Series(array))

### 从字典创建

从字典创建 `Series` 时，字典键会成为索引，字典值会成为数据。

In [ ]:
my_dict = {"a": 50, "b": 60, "c": 70, "d": 80}
series = pd.Series(my_dict)
print(series)
print(series["a"])

### 索引对齐运算

两个 `Series` 做算术运算时，Pandas 会先按索引对齐。找不到匹配索引的位置会得到缺失值 `NaN`。

In [ ]:
series1 = pd.Series([1, 2, 3, 4], ["London", "HongKong", "Shanghai", "Shenzhen"])
series2 = pd.Series([0, 6, 7, 8], ["London", "Shenzhen", "NewYork", "Delhi"])

print(series1 + series2)
print(series1 - series2)
print(series1 * series2)

## Pandas DataFrame 基础

`DataFrame` 是二维表格数据结构，数据按行和列组织。这个主题包括创建表格、重命名列、选择列、增加列、删除行列、行列定位和条件筛选。

### 创建与重命名

`pd.DataFrame(data, index=index, columns=columns)` 可以从数组、字典或其他结构创建表格。

In [ ]:
import numpy as np
import pandas as pd


rng = np.random.default_rng(42)
data = {
    "col one": [100, 200, 300],
    "col two": [400, 500, 600],
    "col three": [700, 800, 900],
}
df = pd.DataFrame(data, index=list("abc"))
print(df)

df = df.rename({"col one": "col_one", "col two": "col_two"}, axis="columns")
df.columns = df.columns.str.replace(" ", "_")
print(df)
print(df.add_prefix("X_"))

### 列选择和新增列

单列选择返回 `Series`，多列选择返回 `DataFrame`。新增列可以来自 `Series`，也可以由已有列计算得到。

In [ ]:
print(df["col_one"])
print(df[["col_one", "col_two"]])

df["col_four"] = pd.Series([1000, 1100, 1200]).values
df["col_one_plus_two"] = df["col_one"] + df["col_two"]
print(df)

### 删除和定位

`.drop()` 默认返回新对象。`.loc[]` 按索引标签定位，`.iloc[]` 按整数位置定位。

In [ ]:
without_extra = df.drop("col_one_plus_two", axis=1)
print(without_extra)
print(df.loc["b"])
print(df.loc[["b", "c"], ["col_one", "col_two"]])
print(df.iloc[[0, 1]])

### 条件筛选

方括号中可以传入布尔条件。多个条件要用 `&`、`|` 和 `~`，每个条件表达式需要括号包裹。

In [ ]:
df = pd.DataFrame(
    data=rng.standard_normal((5, 4)), index=list("abcde"), columns=list("WXYZ")
)
print(df[df["X"] > 0.5])
print(df[(df["W"] < 0) & (df["X"] < 0.5)])
print(df[~((df["W"] > 0) | (df["X"] > 0.5))])

### 重置和设置索引

`.reset_index()` 可以把索引变回普通列或直接丢弃旧索引，`.set_index()` 可以把某一列设为索引。

In [ ]:
reset_df = df.reset_index(drop=True)
reset_df["index"] = list("abcde")
indexed_df = reset_df.set_index("index")
indexed_df.index.name = "mxh"
print(indexed_df)

## Pandas 多级索引

多级索引由多层标签组成，适合表达分组层级。可以通过 `from_arrays()`、`from_tuples()` 或 `from_product()` 创建。

### 创建 MultiIndex

下面用元组列表创建两层索引，并给索引层命名。

In [ ]:
import numpy as np
import pandas as pd


rng = np.random.default_rng(42)
outside = ["level 0", "level 0", "level 0", "level 1", "level 1", "level 1"]
inside = [21, 22, 23, 21, 22, 23]
index = pd.MultiIndex.from_tuples(list(zip(outside, inside)))
df = pd.DataFrame(rng.standard_normal((6, 2)), index=index, columns=["A", "B"])
df.index.names = ["level", "num"]
print(df)

### 分层选择

`.loc[]` 可以逐层选择，`.xs()` 可以按指定 level 做交叉选择。

In [ ]:
print(df.loc["level 0"])
print(df.loc["level 0"].loc[21])
print(df.xs(22, level="num"))

## Pandas 数据合并

Pandas 常见合并方式包括 `concat()`、`merge()` 和 `join()`。选择哪一种取决于你是按轴堆叠、按列键连接，还是按索引连接。

### concat 堆叠

`pd.concat()` 默认按行堆叠。按列堆叠时需要设置 `axis=1`，并注意索引对齐。

In [ ]:
import pandas as pd


df1 = pd.DataFrame([["A0", "B0"], ["A1", "B1"]], columns=["A", "B"])
df2 = pd.DataFrame([["A2", "B2"], ["A3", "B3"]], columns=["A", "B"])
dffull = pd.concat([df1, df2]).reset_index(drop=True)
print(dffull)
print(pd.concat([df1, df2], axis=1))

### merge 归并

`pd.merge(left, right, how="inner", on="key")` 类似 SQL 表连接，常按一个或多个键列合并。

In [ ]:
left = pd.DataFrame(
    {"key": ["K0", "K1", "K2", "K3", "K4"], "A": ["A0", "A1", "A2", "A3", "A4"]}
)
right = pd.DataFrame({"key": ["K0", "K1", "K2", "K3"], "C": ["C0", "C1", "C2", "C3"]})
print(pd.merge(left, right, how="inner", on="key"))
print(pd.merge(left, right, how="outer", on="key"))

### 多键合并和索引连接

多键合并要求所有键同时匹配。`.join()` 使用索引作为公共键。

In [ ]:
left = pd.DataFrame(
    {
        "key1": ["K0", "K0", "K1", "K2"],
        "key2": ["K0", "K1", "K2", "K3"],
        "A": ["A0", "A1", "A2", "A3"],
    }
)
right = pd.DataFrame(
    {
        "key1": ["K0", "K1", "K1", "K2"],
        "key2": ["K0", "K0", "K0", "K0"],
        "C": ["C0", "C1", "C2", "C3"],
    }
)
print(pd.merge(left, right, on=["key1", "key2"], how="outer"))

left_i = pd.DataFrame({"A": ["A0", "A1", "A2"]}, index=["2000", "2001", "2006"])
right_i = pd.DataFrame({"C": ["C0", "C1", "C2"]}, index=["2000", "2007", "2006"])
print(left_i.join(right_i))
print(left_i.join(right_i, how="inner"))

## Pandas 数据读取与清洗

这个主题整理 CSV 读取、数据概览、按类型选列、类型转换、缺失值处理、插值和重复值处理。

### 读取 CSV 与概览

`pd.read_csv()` 可以把 CSV 文件读取为 `DataFrame`。读取后常用 `head()`、`tail()`、`info()` 和 `describe()` 快速了解数据。

In [ ]:
import pandas as pd


df = pd.read_csv("../assets/datasets/titanic.csv", encoding="utf-8")
print(df.head())
print(df.tail())
print(df.describe())

### 根据类型选取列

`select_dtypes()` 可以按数据类型包含或排除列。

In [ ]:
print(df.dtypes)
print(df.select_dtypes(include="number").head())
print(df.select_dtypes(include="object").head())
print(df.select_dtypes(exclude="number").head())

### 类型转换

`astype()` 适合数据能直接转换的场景；包含无效值时，可用 `pd.to_numeric(errors="coerce")` 把无效值转换为 `NaN`。

In [ ]:
raw = pd.DataFrame(
    {
        "col_one": ["1.1", "2.2", "3.3"],
        "col_two": ["4.4", "5.5", "6.6"],
        "col_three": ["7.7", "8.8", "-"],
    }
)
print(raw.astype({"col_one": "float", "col_two": "float"}).dtypes)
print(pd.to_numeric(raw.col_three, errors="coerce"))
print(raw.apply(pd.to_numeric, errors="coerce").fillna(0))

### 缺失值处理

`isna()` 可以统计缺失值，`dropna()` 可以删除包含缺失值的行或列，`interpolate()` 可以插补缺失值。

In [ ]:
df = pd.read_csv("../assets/datasets/titanic.csv")
print(df.isna().sum())
print(df.dropna(axis=1).head())
print(df.dropna(axis=1, thresh=len(df) * 0.90).head())

sample = pd.DataFrame(
    {"A": [12, 4, 5, None, 1], "B": [None, 2, 54, 3, None], "C": [20, 16, None, 3, 8]}
)
print(sample.interpolate(method="linear", limit_direction="forward"))
print(sample.interpolate(method="linear", limit_direction="backward", limit=1))

### 重复值处理

`duplicated()` 标记重复行，`drop_duplicates()` 删除重复行。`subset` 可用于指定参与判断的列。

In [ ]:
dupes = pd.DataFrame(
    {
        "col1": ["one", "one", "two", "two", "two"],
        "col2": [1, 2, 1, 2, 1],
        "col3": ["AA", "BB", "CC", "DD", "EE"],
    }
)
print(dupes.duplicated())
print(dupes.duplicated(subset=["col1"]))
print(dupes.drop_duplicates(subset=["col1"]))

## Pandas 可视化

Pandas 内置的 `.plot()` 方法基于 Matplotlib，适合快速查看 DataFrame 和 Series 的趋势、分布和关系。

### 折线图

`kind="line"` 可绘制折线图，也可以通过 `subplots=True` 分面展示各列。

In [ ]:
import numpy as np
import pandas as pd


rng = np.random.default_rng(42)
df = pd.DataFrame(rng.standard_normal((20, 4)))
df.iloc[:, 2:] *= 100

df.plot(kind="line", marker=".", secondary_y=[2, 3], style=["--", "-.", "-", "-"])
df.plot(
    kind="line", marker=".", subplots=True, layout=(2, 2), style=["--", "-.", "-", "-"]
)

### 散点图和条形图

散点图通过 `x`、`y` 指定横纵轴，条形图支持垂直、水平和堆叠形式。

In [ ]:
d1 = rng.normal(20, 5.5, (100, 2))
d2 = rng.normal(50, 8.5, (100, 2))
scatter_df = pd.DataFrame(np.vstack((d1, d2)), columns=["x", "y"])
scatter_df["s"] = 1500 / np.sqrt((scatter_df.x - 20) ** 2 + (scatter_df.y - 50) ** 2)
scatter_df["c"] = np.sqrt(scatter_df.x**2 + scatter_df.y**2)
scatter_df.plot(kind="scatter", x="x", y="y", s="s", c="c", colormap="rainbow")

bar_df = pd.DataFrame(
    rng.integers(1, 10, (5, 2)), columns=["NW", "SE"], index=list("ABCDE")
)
bar_df.plot(kind="bar", color={"NW": "b", "SE": "r"}, table=True, xticks=[])
bar_df.plot(kind="barh", subplots=True)
bar_df.plot(kind="bar", stacked=True)

### 分布图

直方图、密度图和散点矩阵都适合观察数据分布。散点矩阵需要多列数值数据。

In [ ]:
hist_df = pd.DataFrame(
    {
        "d1": rng.standard_normal(400),
        "d2": rng.normal(10, 2, 400),
        "d3": rng.normal(25, 5, 400),
    }
)
hist_df["type"] = rng.choice(list("ABCD"), size=400)
hist_df.plot(kind="hist", bins=50, alpha=0.85)
hist_df.hist(by="type", bins=20, layout=(2, 2))

pd.DataFrame({"d1": rng.standard_normal(100), "d2": rng.normal(5, 0.5, 100)}).plot(
    kind="density", title="BW:scott"
)

### 饼图和散点矩阵

饼图适合展示占比。`pd.plotting.scatter_matrix()` 可快速查看多变量之间的两两关系。

In [ ]:
pie_df = pd.DataFrame(rng.random((5, 2)), columns=["NW", "SE"], index=list("ABCDE"))
pie_df.plot.pie(y="NW", figsize=(5, 5), autopct="%.2f")
pie_df.plot(kind="pie", subplots=True, figsize=(11, 6), autopct="%.2f")

matrix_df = pd.DataFrame(rng.normal(size=(50, 4)), columns=list("ABCD"))
pd.plotting.scatter_matrix(matrix_df, alpha=0.7, figsize=(7, 7), diagonal="hist")

## Matplotlib 折线图

折线图适合展示连续变量变化趋势。Matplotlib 支持状态式 `pyplot` 接口，也支持更适合复杂图形的面向对象 `Axes` 接口。

### Figure、Axes 与 plot

`Figure` 是图形容器，`Axes` 是实际绘图区。`plot()` 可以绘制单条线或多条线。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


x = np.linspace(0, 10, 100)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)")
ax.legend()

### 颜色、线型和标记

颜色可以使用名称、单字母缩写、灰阶、十六进制或 RGB 元组。线型和颜色也可以合并成简写格式。

In [ ]:
plt.plot(x, np.sin(x - 0), color="blue")
plt.plot(x, np.sin(x - 1), color="g")
plt.plot(x, np.sin(x - 2), color="0.75")
plt.plot(x, np.sin(x - 3), color="#FFDD44")
plt.plot(x, np.sin(x - 4), color=(1.0, 0.2, 0.3))
plt.plot(x, np.sin(x - 5), color="chartreuse")

In [ ]:
x_short = np.linspace(0, 10, 20)
plt.plot(x_short, x_short + 0, "-g")
plt.plot(x_short, x_short + 1, "--c")
plt.plot(x_short, x_short + 2, "-.k")
plt.plot(x_short, x_short + 3, ":r")
plt.plot(
    x_short,
    x_short + 4,
    color="red",
    marker="*",
    markerfacecolor="white",
    markeredgecolor="gray",
)

### 坐标轴范围和标签

`xlim()`、`ylim()` 和 `axis()` 可以控制坐标范围。标题、坐标轴标签和图例能帮助读者理解图表含义。

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, np.sin(x))
ax.set(xlim=(0, 10), ylim=(-2, 2), xlabel="x", ylabel="sin(x)", title="A Simple Plot")
ax.grid(True)

## Matplotlib 坐标轴、刻度与标注

折线图之外，图表质量往往取决于坐标轴位置、刻度格式和文本标注是否清楚。

### 坐标轴位置

`Axes.spines` 可以控制绘图区边框线的位置、颜色和可见性。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


fig, ax = plt.subplots()
x = np.linspace(-np.pi, np.pi, 100)
ax.plot(x, np.sin(x))
ax.spines["left"].set_position(("data", 0))
ax.spines["bottom"].set_position(("data", 0))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

### 双 y 轴

不同尺度的数据可以通过 `twinx()` 共享 x 轴并使用两个 y 轴展示。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax_twin = ax.twinx()

x = np.linspace(1, 10, 50)
height = np.linspace(100, 200, 50)
weight = np.linspace(30, 80, 50)
ax.plot(x, height, "b-")
ax_twin.plot(x, weight, "r--")
ax.set_ylabel("Height(cm)", color="blue")
ax_twin.set_ylabel("Weight(kg)", color="red")

### 刻度格式

`set_xticks()`、`MultipleLocator` 和 `FuncFormatter` 可以控制刻度位置和显示文本。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(0, 2 * np.pi, 1000)
ax.plot(x, np.sin(x), lw=3)
ax.grid(True)
ax.set_xlim(0, 2 * np.pi)
ax.xaxis.set_major_locator(plt.MultipleLocator(np.pi / 2))


def format_func(value, tick_number):
    n = int(np.round(2 * value / np.pi))
    if n == 0:
        return "0"
    if n == 1:
        return r"$\frac{\pi}{2}$"
    if n == 2:
        return r"$\pi$"
    if n % 2:
        return rf"$\frac{{{n}\pi}}{{2}}$"
    return rf"${n // 2}\pi$"


ax.xaxis.set_major_formatter(plt.FuncFormatter(format_func))

### 文本和箭头标注

`text()` 用于放置文字，`annotate()` 可以创建箭头标注。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
x = np.linspace(0, 20, 1000)
ax.plot(x, np.cos(x))
ax.axis([0, 20, -2, 2])
ax.text(x=np.pi, y=-1.1, s=r"$\cos \pi = -1$", size=12, color="red", ha="left")
ax.annotate(
    "local maximum",
    xy=(2 * np.pi, 1),
    xytext=(10, 1.5),
    arrowprops={"color": "blue", "width": 1, "headwidth": 8},
)

## Matplotlib 子图

子图用于在一个 Figure 中并列展示多个 Axes，便于比较不同图表或不同变量。

### subplots

`plt.subplots()` 是最常用的子图创建方法，可以指定行列数、共享坐标轴和整体尺寸。

In [ ]:
import itertools
import matplotlib.pyplot as plt
import numpy as np


fig, axes = plt.subplots(2, 3, sharex="all", sharey="all")
for i, j in itertools.product(range(2), range(3)):
    axes[i, j].text(0.5, 0.5, str((i, j)), fontsize=18, ha="center")

### subplot_mosaic

`plt.subplot_mosaic()` 可以用二维布局描述不同大小的子图区域。

In [ ]:
fig, axes = plt.subplot_mosaic(
    [["upper left", "right"], ["lower left", "right"]],
    figsize=(5.5, 3.5),
    layout="constrained",
)
for name, ax in axes.items():
    ax.text(
        0.5,
        0.5,
        f'ax["{name}"]',
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
    )
fig.suptitle("plt.subplot_mosaic()")

### 子图间距

`tight_layout()` 可以自动调整子图之间的距离，避免标题和标签重叠。

In [ ]:
fig, axes = plt.subplots(2, 2)
x = np.linspace(-np.pi, np.pi, 100)
axes[0, 0].plot(x, np.sin(x))
axes[0, 0].set_title("sin(x)")
axes[0, 1].plot(x, np.cos(x))
axes[0, 1].set_title("cos(x)")
axes[1, 0].plot(x, np.sin(x - 1))
axes[1, 0].set_title("sin(x-1)")
axes[1, 1].plot(x, np.cos(x - 1))
axes[1, 1].set_title("cos(x-1)")
plt.tight_layout(pad=2, w_pad=1, h_pad=2)

## Matplotlib 散点图

散点图用于展示两个变量之间的关系。`plt.scatter()` 可以为每个点分别设置大小、颜色、透明度和颜色映射。

### 基础散点图

`x` 和 `y` 需要长度一致，`marker` 控制点样式。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


x = np.linspace(0, 10, 50)
y = np.sin(x)
plt.scatter(x, y, marker="o")

### 颜色、大小和色条

当颜色或大小来自数据时，散点图可以同时编码更多变量。`colorbar()` 用于展示颜色映射含义。

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(100)
y = rng.standard_normal(100)
colors = rng.random(100)
sizes = 1000 * rng.random(100)

plt.scatter(x, y, c=colors, s=sizes, alpha=0.3, cmap="viridis")
plt.colorbar()

### 多维数据展示

散点位置、大小和颜色可以分别绑定不同变量，用于快速观察样本分布。

In [ ]:
rng = np.random.default_rng(42)
features = rng.normal(size=(4, 120))
target = rng.integers(0, 3, size=120)

fig, ax = plt.subplots()
scatter = ax.scatter(
    features[0],
    features[1],
    alpha=0.5,
    s=100 * np.abs(features[3]),
    c=target,
    cmap="viridis",
)
ax.set_xlabel("feature 0")
ax.set_ylabel("feature 1")
ax.legend(*scatter.legend_elements(), loc="upper left", title="Classes")

## Matplotlib 柱形图

柱形图用于比较离散类别的数值大小。`bar()` 绘制垂直柱形图，`barh()` 绘制水平柱形图。

### 基础柱形图

`x` 指定类别或位置，`height` 指定高度，`align` 控制对齐方式。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


data = {"C": 20, "C++": 15, "Java": 30, "Python": 35}
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(list(data.keys()), list(data.values()), color="blue", align="edge")
ax.set_xlabel("Courses offered")
ax.set_ylabel("No. of students enrolled")
ax.set_title("Students enrolled in different courses")

### 正负柱形图

通过正负方向可以对比两组数据，也可以给柱形添加文字标签。

In [ ]:
rng = np.random.default_rng(42)
n = 12
x = np.arange(n)
y1 = (1 - x / float(n)) * rng.uniform(0.5, 1.0, n)
y2 = (1 - x / float(n)) * rng.uniform(0.5, 1.0, n)

fig, ax = plt.subplots(figsize=(10, 6))
bar1 = ax.bar(x, y1, facecolor="#9999ff", edgecolor="white")
bar2 = ax.bar(x, -y2, facecolor="#ff9999", edgecolor="white")
ax.set_ylim(-1.1, 1.1)
for p1, p2 in zip(bar1.patches, bar2.patches):
    ax.text(
        p1.get_x() + p1.get_width() / 2,
        p1.get_height() + 0.05,
        f"{p1.get_height():.2f}",
        ha="center",
    )
    ax.text(
        p2.get_x() + p2.get_width() / 2,
        p2.get_height() - 0.1,
        f"{p2.get_height():.2f}",
        ha="center",
    )

### 水平柱形图

水平柱形图常用于类别名称较长或需要横向比较的场景。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(-1, 1)
bar1 = ax.barh(x, y1, facecolor="#9999ff", edgecolor="white")
bar2 = ax.barh(x, -y2, facecolor="#ff9999", edgecolor="white")
for p1, p2 in zip(bar1.patches, bar2.patches):
    ax.text(
        p1.get_width(),
        p1.get_y() + p1.get_height() / 3,
        f"{p1.get_width():.2f}",
        ha="left",
    )
    ax.text(
        p2.get_width(),
        p2.get_y() + p2.get_height() / 3,
        f"{p2.get_width():.2f}",
        ha="right",
    )

## Matplotlib 直方图

直方图用于展示数值分布。`hist()` 会把数据分桶并绘制每个桶的频数或密度。

### 基础直方图

`bins` 控制分桶数量，`density=True` 可以显示概率密度。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


rng = np.random.default_rng(42)
data = rng.standard_normal(1000)
plt.hist(
    data,
    bins=30,
    density=True,
    alpha=0.5,
    histtype="stepfilled",
    color="steelblue",
    edgecolor="none",
)

### 多个分布对比

`histtype="stepfilled"` 和透明度适合叠加多个分布。

In [ ]:
x1 = rng.normal(0, 0.8, 1000)
x2 = rng.normal(-2, 1, 1000)
x3 = rng.normal(3, 2, 1000)
kwargs = {"histtype": "stepfilled", "alpha": 0.3, "density": True, "bins": 40}
plt.hist(x1, **kwargs)
plt.hist(x2, **kwargs)
plt.hist(x3, **kwargs)

### 只计算分桶

如果只需要桶计数而不绘图，可以使用 `np.histogram()`。

In [ ]:
counts, bin_edges = np.histogram(data, bins=5)
print(counts)
print(bin_edges)

## Matplotlib 饼图

饼图用于展示部分与整体的占比。`pie()` 支持标签、突出扇区、百分比文本、阴影和图例。

### 基础饼图

`explode` 可以让指定扇形偏离圆心，`autopct` 控制百分比文本格式。

In [ ]:
import matplotlib.pyplot as plt


labels = "Frogs", "Hogs", "Dogs", "Logs"
sizes = [15, 30, 45, 10]
explode = (0, 0.1, 0, 0)

fig, ax = plt.subplots()
wedges, texts, autotexts = ax.pie(
    sizes, explode=explode, labels=labels, autopct="%0.1f%%", shadow=True, startangle=90
)
ax.axis("equal")
ax.legend(title="Ingredients", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

## Matplotlib 误差可视化

误差可视化用于展示估计值的不确定性。基础场景用误差条，连续区间可以用填充带。

### 基础误差条

`errorbar()` 的 `fmt` 参数控制线条和点样式，`yerr` 或 `xerr` 控制误差范围。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


rng = np.random.default_rng(42)
x = np.linspace(0, 10, 30)
y = np.sin(x)
error = 0.1 * rng.standard_normal(30)
plt.errorbar(
    x,
    y,
    yerr=np.abs(error),
    fmt="-o",
    ecolor="lightblue",
    color="green",
    elinewidth=3,
    capsize=4,
)

### 连续误差带

对于连续误差范围，可以用 `plot()` 绘制中心线，再用 `fill_between()` 填充上下界之间的区域。

In [ ]:
xfit = np.linspace(0, 10, 1000)
yfit = xfit * np.sin(xfit)
dyfit = 0.2 + 0.1 * np.abs(np.cos(xfit))

plt.plot(xfit, yfit, "-", color="gray")
plt.fill_between(x=xfit, y1=yfit - dyfit, y2=yfit + dyfit, color="gray", alpha=0.2)
plt.xlim(0, 10)

## Matplotlib 轮廓图

轮廓图用于在二维平面上展示三维函数值，例如等高线地图、温度分布和决策边界。

### contour 和 contourf

`contour()` 绘制轮廓线，`contourf()` 绘制填充轮廓图。网格数据通常由 `np.meshgrid()` 生成。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def f(x, y):
    return np.sin(x) ** 10 + np.cos(10 + y * x) * np.cos(x)


x = np.linspace(0, 5, 50)
y = np.linspace(0, 5, 40)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig, ax = plt.subplots(figsize=(10, 6))
contour = ax.contour(X, Y, Z, levels=8, cmap="jet")
ax.clabel(contour, fontsize=10, colors="k")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
contour = ax.contourf(X, Y, Z, levels=20, cmap="jet")
plt.colorbar(contour)

### imshow

`imshow()` 可以直接把二维数值数组映射为图像。下面用 Mandelbrot 集做示例。

In [ ]:
def compute_mandelbrot(max_iter, threshold, nx, ny):
    x = np.linspace(-2, 1, nx)
    y = np.linspace(-1.5, 1.5, ny)
    c = x[:, None] + 1j * y[None, :]
    z = np.zeros_like(c)
    mandelbrot_set = np.zeros((nx, ny))
    for step in range(max_iter):
        z = z**2 + c
        mask = abs(z) > threshold
        mandelbrot_set[mask] = step
        c[mask] = 0 + 0j
    return mandelbrot_set


mandelbrot_set = compute_mandelbrot(50, 2.0, 300, 300)
plt.imshow(mandelbrot_set.T, extent=[-2, 1, -1.5, 1.5], cmap="jet")
plt.colorbar()

## Matplotlib 三维图形

Matplotlib 可以通过 `projection="3d"` 创建三维坐标轴。现代 Matplotlib 会自动注册 3D projection，通常不需要显式导入 `mpl_toolkits.mplot3d`。三维图适合展示空间曲线、散点、三维柱形、轮廓、框线和表面。

### 三维坐标轴

使用 `projection="3d"` 创建三维 Axes。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


fig = plt.figure()
ax = plt.axes(projection="3d")

### 三维点和线

`plot3D()` 绘制三维折线，`scatter3D()` 绘制三维散点。

In [ ]:
rng = np.random.default_rng(42)
ax = plt.axes(projection="3d")
zline = np.linspace(0, 15, 1000)
xline = np.sin(zline)
yline = np.cos(zline)
ax.plot3D(xline, yline, zline, "gray")

zdata = 15 * rng.random(100)
xdata = np.sin(zdata) + 0.1 * rng.standard_normal(100)
ydata = np.cos(zdata) + 0.1 * rng.standard_normal(100)
ax.scatter3D(xdata, ydata, zdata, c=zdata, cmap="Dark2")

### 三维柱形图

`bar3d()` 使用三维位置和长宽高绘制柱体。

In [ ]:
ax = plt.axes(projection="3d")
x = np.arange(10)
y = np.arange(10)
z = np.zeros(10)
dx = np.ones(10)
dy = np.ones(10)
dz = np.arange(10)
ax.bar3d(x, y, z, dx, dy, dz, color="b", alpha=0.8)

### 三维轮廓和表面

网格数据可以绘制三维轮廓图、框线图和表面图。

In [ ]:
def f(x, y):
    return np.sin(np.sqrt(x**2 + y**2))


x = np.linspace(-6, 6, 30)
y = np.linspace(-6, 6, 30)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

fig = plt.figure()
ax = plt.axes(projection="3d")
ax.contour3D(X, Y, Z, 50, cmap="binary")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(60, 35)

In [ ]:
fig = plt.figure()
ax = plt.axes(projection="3d")
ax.plot_wireframe(X, Y, Z, color="black")
ax.set_title("wireframe")

fig = plt.figure()
ax = plt.axes(projection="3d")
ax.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap="viridis", edgecolor="none")
ax.set_title("surface")

## scikit-learn 现代建模流程

这个 notebook 用 Titanic 示例数据演示当前更推荐的机器学习工作流：把数据拆分、缺失值处理、类别编码、数值缩放、模型训练和评估放进可复用的 `Pipeline`。

### 为什么用 Pipeline

直接在完整数据上做预处理再切分，容易把验证集信息泄漏进训练过程。`Pipeline` 和 `ColumnTransformer` 可以把预处理步骤绑定到模型上，让交叉验证和线上推理都走同一套流程。

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../assets/datasets/titanic.csv")

titanic = pd.read_csv(DATA_PATH)
titanic.head()

### 选择特征和目标

这里保留少量结构化特征，重点展示流程。真实项目中应结合业务含义、数据质量和偏差风险重新做特征选择。

In [ ]:
target = "Survived"
numeric_features = ["Pclass", "Age", "Fare"]
categorical_features = ["Sex", "Embarked"]
features = numeric_features + categorical_features

X = titanic[features]
y = titanic[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

### 构建预处理和模型管道

数值列使用中位数补缺失值并标准化；类别列使用众数补缺失值并做 one-hot 编码。`handle_unknown="ignore"` 可以避免预测阶段遇到新类别时报错。

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1_000)),
    ]
)

model

### 交叉验证

交叉验证直接作用在完整管道上，避免每折训练和验证之间的数据泄漏。

In [ ]:
scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=5,
    scoring=["accuracy", "roc_auc"],
)

pd.DataFrame(scores).agg(["mean", "std"]).T

### 在测试集上评估

测试集只在最终评估阶段使用。模型训练仍然调用同一个 `Pipeline`，预测时会自动执行相同的预处理步骤。

In [ ]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"ROC AUC: {roc_auc_score(y_test, probabilities):.3f}")
print(classification_report(y_test, predictions))

### 下一步

可以继续把 `classifier` 替换为 `RandomForestClassifier`、`HistGradientBoostingClassifier` 或其他估计器；只要保持同一个 `Pipeline` 接口，训练、交叉验证和推理代码都不需要大改。